# Figure 1 (b, c, e, f, g) and the plasticity-off control (Supp. Methods S1)

b: 3-D trajectory; c: traces; e, f: learning curves (PC, small and large); g: operating-point scores with 95% bootstrap CIs. The `beta = 0` (frozen) scores printed in the Fig. 1g cell are the plasticity-off control numbers quoted in Supplementary Methods S1.

In [ ]:
from ssb.utils import batch_inits
from ssb.envs import bandits
from ssb.scripts import run
from ssb.models import ssb, rules
from ssb.analysis.best_params import best_params
from importlib import reload
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.family'] = 'Arial'

RULES = ['BCM','OJA','LC','PC','COV','HZ','HR']


In [ ]:
def run_sim(rule, timesteps=1000, alpha=.001, beta=.005, eta_a=.005, error=False):
    hps = batch_inits.make_default_hps(53)
    hps['alpha'] = jnp.array([alpha])
    hps['beta'] = jnp.array([beta])
    hps['eta_a'] = jnp.array([eta_a])

    update_weights, update_states = rules.rule2fns(rule)
    acts, states, weights, tern, key = batch_inits.init_params(hps)
    acts = jnp.zeros_like(acts).at[1:].set(1.)
    reward = jnp.array([1.])

    acts_log    = np.zeros((timesteps, 3))
    weights_log = np.zeros((timesteps, 2))
    errors_log  = np.zeros((timesteps, 2)) if error else None

    for t in range(timesteps):
        if t > timesteps // 2:
            reward = jnp.array([0.])
        stim   = ssb.expand_stimulus(reward, acts)
        acts   = ssb.update_acts(stim, acts, states, weights, tern, hps)
        acts, key = ssb.act_noise(acts, key, hps)
        states = update_states(acts, states, hps)
        weights = update_weights(stim, acts, states, weights, tern, hps)
        acts_log[t]    = acts
        weights_log[t] = weights[0, 1:]
        if error:
            if rule == 'PC':
                errors_log[t] = (1 - weights[0, 1:] * acts[0] / (acts[1:] + 1e-8)) * acts[1:]
            elif rule == 'HR':
                errors_log[t] = (reward - acts[0]) * acts[1:]

    return (acts_log, weights_log, errors_log) if error else (acts_log, weights_log)

In [ ]:
def style_ax(ax, fontsize=8):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
    ax.tick_params(axis="both", labelsize=fontsize, length=2, pad=1.5)
    ax.yaxis.set_tick_params(width=0.5)
    ax.xaxis.set_tick_params(width=0.5)


C_STIM   = "#444444"
C_ACT    = "#2171b5"
C_PRED   = "#6baed6"
C_WEIGHT = "#08519c"


def plot_traces(rule, timesteps=1000, alpha=.01, beta=.01, eta_a=.005,
                ylim=(-.1, 1.1), save=None):
    acts, weights, errors = run_sim(rule, timesteps=timesteps,
                                    alpha=alpha, beta=beta, eta_a=eta_a, error=True)
    stim = np.ones(timesteps); stim[timesteps // 2:] = 0
    t = np.arange(timesteps)
    fig, axes = plt.subplots(1, 3, figsize=(2.93, 0.65))
    lw = 0.8
    axes[0].plot(t, stim,                         color=C_STIM,   lw=lw)
    axes[0].plot(t, acts[:, 0],                   color=C_ACT,    lw=lw)
    axes[1].plot(t, acts[:, 1],                   color=C_ACT,    lw=lw)
    axes[1].plot(t, acts[:, 0] * weights[:, 0],   color=C_PRED,   lw=lw)
    axes[2].plot(t, weights[:, 0],                color=C_WEIGHT, lw=lw)
    ylabels = ["Activity", "Activity", "Weight"]
    for i, ax in enumerate(axes):
        style_ax(ax)
        ax.set_xlim(0, timesteps)
        ax.set_xlabel("Timestep", fontsize=8)
        ax.set_ylabel(ylabels[i], fontsize=8)
        if i < 2:
            ax.set_ylim(*ylim)
            ax.set_yticks([0, 0.5, 1])
    fig.tight_layout(pad=0.3)
    if save:
        plt.savefig(save, dpi=300, bbox_inches="tight")
    return acts, weights, errors


def plot_error(errors, save=None):
    fig, ax = plt.subplots(figsize=(1.4, 0.62))
    ax.plot(errors[:, 0], color=C_ACT, lw=0.8)
    ax.axhline(0, color="#aaaaaa", lw=0.5, linestyle="--")
    style_ax(ax)
    ax.set_xlabel("Timestep", fontsize=8)
    ax.set_ylabel("Error", fontsize=8)
    fig.tight_layout(pad=0.3)
    if save:
        plt.savefig(save, dpi=300, bbox_inches="tight")


def rolling_mean(x, w):
    n = (len(x) // w) * w
    return x[:n].reshape(-1, w).mean(axis=1)


def plot_reward_curve_multiple(ax, actions, window, line_alpha=1.0, fill_alpha=0.3,
                                color="blue", label=None, error=False):
    """Plot median (± IQR if error=True) onto an existing axis. Caller controls ylim/styling."""
    smoothed = np.stack([rolling_mean(actions[:, i], window)
                         for i in range(actions.shape[1])], axis=1)
    median = np.median(smoothed, axis=1)
    p25    = np.percentile(smoothed, 25, axis=1)
    p75    = np.percentile(smoothed, 75, axis=1)
    t      = np.arange(smoothed.shape[0]) * window
    ax.plot(t, median, alpha=line_alpha, color=color, lw=0.8, label=label)
    if error:
        ax.fill_between(t, p25, p75, alpha=fill_alpha, color=color, linewidth=0)


def plot_reward_curve(actions, window, figsize=(1.3, 0.68), ylim=(-.05, 1.1), save=None):
    fig, ax = plt.subplots(figsize=figsize)
    plot_reward_curve_multiple(ax, actions, window, color=C_ACT, fill_alpha=0.25, error=True)
    ax.set_xlabel("Timestep", fontsize=8)
    ax.set_ylabel("P(optimal)", fontsize=8)
    ax.set_yticks([0, 0.5, 1])
    if ylim:
        ax.set_ylim(*ylim)
    style_ax(ax)
    fig.tight_layout(pad=0.3)
    if save:
        plt.savefig(save, dpi=300, bbox_inches="tight")


# Fig. 1b, c: Tiny network as dynamical system

In [ ]:
acts_pc, weights_pc, errors_pc = plot_traces('PC', ylim=(-.1, 1.1), save='f1_pc_noisy.svg')
plot_error(errors_pc, save='f1_pc_error.svg')

## Fig. 1b: 3-D trajectory of the run in Fig. 1c

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.signal import savgol_filter
from mpl_toolkits.mplot3d import Axes3D

# ── Custom Smooth Gradients (Extracted from your image palette) ─────────────
custom_pink = mcolors.LinearSegmentedColormap.from_list(
    'custom_pink', ['#f8e3e5', '#d86577', '#692530']
)
custom_yellow = mcolors.LinearSegmentedColormap.from_list(
    'custom_yellow', ['#fdf3e2', '#f1b457', '#8e540b']
)

def plot_3d_trajectory(acts, weights,
                       cmap1=custom_pink, cmap2=custom_yellow,
                       smooth_window=25, smooth_poly=5,
                       n_arrows=0, arrow_scale=0.04,
                       figsize=(5, 5),
                       azim=-60, elev=30):
    """
    Plot a 3D trajectory (x, y, z) split into two halves with custom colormaps,
    including a drop shadow on the bottom pane.
    
    Parameters
    ----------
    acts    : (N, 2+) array  — columns 0 and 1 are x and y
    weights : (N, 1+) array  — column 0 is z
    cmap1   : colormap name or object for first half
    cmap2   : colormap name or object for second half
    smooth_window : Savitzky-Golay window length (odd integer, < N)
    smooth_poly   : Savitzky-Golay polynomial order
    n_arrows      : number of direction arrows per half
    arrow_scale   : quiver arrow length scale (tune to your data range)
    figsize       : Tuple defining the figure dimensions
    azim          : Azimuthal viewing angle
    elev          : Elevation viewing angle
    """
    x_raw = acts[:, 0]
    y_raw = weights[:, 0]
    z_raw = acts[:, 1]
    N = len(x_raw)

    # ── Smooth ──────────────────────────────────────────────────────────────
    def smooth(v):
        return savgol_filter(v, window_length=smooth_window, polyorder=smooth_poly)

    x = smooth(x_raw)
    y = smooth(y_raw)
    z = smooth(z_raw)

    # ── Arc-length parameterisation (gives uniform colour spacing) ──────────
    dx, dy, dz = np.diff(x), np.diff(y), np.diff(z)
    seg_len = np.sqrt(dx**2 + dy**2 + dz**2)
    arc = np.concatenate([[0], np.cumsum(seg_len)])
    arc /= arc[-1]          # normalise to [0, 1]

    # ── Split at midpoint index ─────────────────────────────────────────────
    mid = N // 2

    # ── Figure setup ────────────────────────────────────────────────────────
    fig = plt.figure(figsize=figsize)
    ax  = fig.add_subplot(111, projection='3d')

    # Minimal pane styling
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = True

    # ── Grid Styling & Explicit Ticks ───────────────────────────────────────
    ax.grid(True)
    for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
        axis._axinfo['grid']['color'] = 'lightgray'
        axis._axinfo['grid']['linestyle'] = ':'
        axis._axinfo['grid']['linewidth'] = .5
        
    # Set explicit user-defined ticks
    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0.7, 0.75, 0.8])
    ax.set_zticks([0, 0.5, 1])
    
    # Define the floor level for the shadow based on the lowest z-tick
    z_floor = 0 

    ax.tick_params(labelsize=7)
    for spine in ax.spines.values():
        spine.set_visible(False)

    # ── Helper: draw a coloured segment + shadows + points ──────────────────
    def draw_half(idx_start, idx_end, cmap_name, label):
        sl = slice(idx_start, idx_end + 1)
        xs, ys, zs = x[sl], y[sl], z[sl]

        ax_arc = arc[sl]
        t = (ax_arc - ax_arc[0]) / (ax_arc[-1] - ax_arc[0] + 1e-12)

        # Robust handling for both Colormap objects and string identifier names
        cmap = cmap_name if isinstance(cmap_name, mcolors.Colormap) else plt.get_cmap(cmap_name)
        n    = len(xs)

        # Draw the shadow for the line path
        ax.plot(xs, ys, np.full_like(zs, z_floor), 
                color='black', alpha=0.05, linewidth=1.5, zorder=1)

        # Draw the main coloured line segments
        for i in range(n - 1):
            col = cmap(0.25 + 0.75 * t[i])
            ax.plot(xs[i:i+2], ys[i:i+2], zs[i:i+2],
                    color=col, linewidth=1.8, solid_capstyle='round',
                    alpha=0.85, zorder=3)

        # Draw shadows for the start and end dots
        ax.scatter(xs[0], ys[0], z_floor, color='black', alpha=0.05, s=20, zorder=1)
        ax.scatter(xs[-1], ys[-1], z_floor, color='black', alpha=0.05, s=20, zorder=1)

        # Draw the main start and end dots
        ax.scatter(*[[v[0]] for v in [xs, ys, zs]],
                   color=cmap(0.3), s=20, zorder=5, edgecolors='w', linewidths=0.5)
        ax.scatter(*[[v[-1]] for v in [xs, ys, zs]],
                   color=cmap(0.9), s=20, zorder=5, edgecolors='w', linewidths=0.5)

    draw_half(0,   mid, cmap1, f'First half')
    draw_half(mid, N-1, cmap2, f'Second half')

    # ── Colorbars (Automatically positioned to the side) ────────────────────
    
    # First half colorbar
    cmap1_obj = cmap1 if isinstance(cmap1, mcolors.Colormap) else plt.get_cmap(cmap1)
    sm1 = plt.cm.ScalarMappable(cmap=cmap1_obj, norm=mcolors.Normalize(0, 1))
    sm1.set_array([])
    cb1 = fig.colorbar(sm1, ax=ax, shrink=0.6, pad=0.1)
    cb1.set_ticks([])
    cb1.ax.set_ylabel(f'First half', fontsize=7, rotation=270, labelpad=10)
    cb1.outline.set_visible(False)

    # Second half colorbar
    cmap2_obj = cmap2 if isinstance(cmap2, mcolors.Colormap) else plt.get_cmap(cmap2)
    sm2 = plt.cm.ScalarMappable(cmap=cmap2_obj, norm=mcolors.Normalize(0, 1))
    sm2.set_array([])
    cb2 = fig.colorbar(sm2, ax=ax, shrink=0.6, pad=0.1)
    cb2.set_ticks([])
    cb2.ax.set_ylabel(f'Second half', fontsize=7, rotation=270, labelpad=10)
    cb2.outline.set_visible(False)

    # ── Labels & Final View ─────────────────────────────────────────────────
    ax.set_xlabel('x (acts[:,0])', fontsize=7, labelpad=3)
    ax.set_ylabel('y (weights[:,0])', fontsize=7, labelpad=3) 
    ax.set_zlabel('z (acts[:,1])', fontsize=7, labelpad=3)
    
    # Ensure the axis limits don't get thrown off by the shadow
    ax.set_zlim(bottom=z_floor)

    ax.view_init(elev=elev, azim=azim)

    fig.set_layout_engine('constrained')
    
    return fig, ax

import os; os.makedirs('svgs', exist_ok=True)
plot_3d_trajectory(acts_pc, weights_pc, azim=30, elev=30)
plt.savefig('svgs/f1_3dtrajpc.svg')

# Fig. 1e, f: Networks seek reward, PC example

In [ ]:
timesteps = 100_000
window    = 100

for size in ('small', 'big'):
    n_seeds = 50
    for rule in ['PC']:
        eta, alpha, beta, tau = best_params(rule, size)
        actions, _, _ = run.run_quick(rule, alpha=alpha, beta=beta, eta_a=eta,
                                      env=bandits.bandit_det, timesteps=timesteps,
                                      size=size, n_seeds=n_seeds, tau=tau)
        plot_reward_curve(actions, window=window, save=f'f1_{rule.lower()}_{size}_reward.svg')
        print(f'size: {size}, eta: {eta}, alpha: {alpha}, beta: {beta}')

# Fig. 1g: reward seeking at the operating points
Plots the parameter-selection score itself (as in `best_params`): the median over steps 80k–100k of P(optimal) averaged across seeds (50 seeds, fresh `bandit_det` run). 
Error bars are 95% bootstrap CIs over seeds (1,000 resamples). Grey dashes (second SVG): the same score with β = 0 (frozen weights), a no-learning reference.

In [ ]:
def compute_dets_sel(size, timesteps=100_000, n_seeds=50, frozen=False, n_boot=1000):
    """Per rule: the parameter-selection score itself, i.e. the median over steps 80k-100k of
    the across-seed mean P(optimal) trace (as in best_params), with a 95% bootstrap CI over
    seeds (seeds resampled with replacement, score recomputed)."""
    rng = np.random.default_rng(0)
    dets = np.zeros((len(RULES), 3))
    for i, rule in enumerate(RULES):
        eta, alpha, beta, tau = best_params(rule, size)
        log = run.run_quick(rule=rule, alpha=alpha, beta=0. if frozen else beta, eta_a=eta,
                            env=bandits.bandit_det, timesteps=timesteps,
                            size=size, seed=235, n_seeds=n_seeds, tau=tau)
        ch = np.asarray(log[0][timesteps // 5 * 4:, :], dtype=float)   # steps x seeds
        score = np.median(ch.mean(axis=1))
        boot = [np.median(ch[:, rng.integers(0, n_seeds, n_seeds)].mean(axis=1)) for _ in range(n_boot)]
        dets[i] = (score, *np.percentile(boot, [2.5, 97.5]))
    return dets


def plot_dets_sel(dets_small, dets_big, frozen_small=None, frozen_big=None, save=None):
    """Same layout as plot_dets_mean; points = selection score, bars = 95% bootstrap CI."""
    x = np.arange(len(RULES)); dx = 0.14
    fig, ax = plt.subplots(figsize=(2.1, 0.75))
    ax.axhline(0.5, color="#aaaaaa", lw=0.4, linestyle="--", zorder=0)
    for d, fr, off, face, lab in [(dets_small, frozen_small, -dx, 'white', 'Small network'),
                                  (dets_big, frozen_big, dx, C_ACT, 'Large network')]:
        if fr is not None:
            ax.hlines(fr[:, 0], x + off - 0.11, x + off + 0.11, color='#9a9a9a', lw=0.8, zorder=1)
        yerr = np.vstack([d[:, 0] - d[:, 1], d[:, 2] - d[:, 0]])
        ax.errorbar(x + off, d[:, 0], yerr=yerr, fmt='o', markersize=2.4, markerfacecolor=face,
                    markeredgecolor=C_ACT, markeredgewidth=0.6, capsize=1.2, capthick=0.5,
                    elinewidth=0.5, color=C_ACT, ecolor=C_ACT, zorder=3, label=lab)
    ax.set_xticks(x); ax.set_xticklabels(RULES, fontsize=8)
    ax.set_ylabel("P(optimal)", fontsize=8)
    ax.set_ylim(0, 1.05); ax.set_yticks([0, 0.5, 1])
    ax.legend(fontsize=6, frameon=False, ncol=2, loc='lower center', bbox_to_anchor=(0.5, 1.0),
              handletextpad=0.2, columnspacing=0.8)
    style_ax(ax)
    fig.tight_layout(pad=0.3)
    if save:
        plt.savefig(save, dpi=300, bbox_inches="tight")
    return fig


In [ ]:
dets_sel = {size: compute_dets_sel(size) for size in ('small', 'big')}
dets_sel_frozen = {size: compute_dets_sel(size, frozen=True) for size in ('small', 'big')}

print(f"{'rule':5s} {'size':6s} {'score':>6s} {'95% CI':>14s} {'frozen':>7s}")
for size in ('small', 'big'):
    for i, rule in enumerate(RULES):
        s, lo, hi = dets_sel[size][i]
        print(f"{rule:5s} {size:6s} {s:6.3f}  [{lo:5.3f}, {hi:5.3f}] {dets_sel_frozen[size][i,0]:7.3f}")

plot_dets_sel(dets_sel['small'], dets_sel['big'], save='svgs/f1_dets_sel.svg')
plt.show()
